# Gemma 4 declarative agent: v3 reproducibility artifact
Kaggle API verified on 2026-11-12: v2 scored 0.18; v3 scored 0.15.
This notebook reproduces v3, which is not the best observed configuration.
Compilation validates configuration only; it does not demonstrate repair success.
The archived v2 bundle remains the best verified candidate.


## 1. Materialize Declarative Submission Bundle


In [ ]:
import hashlib
import json
import re
import shutil
import subprocess
import sys
import zipfile
from pathlib import Path
import yaml

WORKING = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path("working")
WORKING.mkdir(parents=True, exist_ok=True)
BUNDLE = WORKING / "submission_bundle"
shutil.rmtree(BUNDLE, ignore_errors=True)
BUNDLE.mkdir(parents=True, exist_ok=True)

SUBMISSION_FILES = json.loads('{\n  "agent.yaml": "name: swe_agent\\nmodel: gemma-4-31b-it-qat-w4a16-ct\\ndescription: Resolves a repository issue with a minimal, verified source patch.\\ninstruction: !include prompts/system.md\\ntools:\\n  - run_command\\n  - read_file\\n  - edit_file\\n  - write_file\\n  - get_status\\n  - submit_patch\\ngenerate_content_config: !include configs/sampling.yaml\\n",\n  "configs/sampling.yaml": "temperature: 0.4\\ntop_p: 0.95\\ntop_k: 64\\nmax_output_tokens: 8192\\nthinking_config:\\n  thinking_budget: 4096\\n  include_thoughts: true\\n",\n  "eval_config.yaml": "evaluation:\\n  timeout_seconds: 240\\n  max_tool_calls: 28\\n  max_time_minutes: 8\\n  max_turns: 80\\n",\n  "prompts/system.md": "You are an autonomous software engineer. Make the smallest correct fix to SOURCE code in /workspace so hidden tests pass. Treat exact symbol names, types, exception classes, and messages in the issue as literal contracts.\\n\\n# HARD LIMITS\\n- Budget: 28 tool calls and 8 minutes (`submit_patch` and `get_status` are free). When either expires, `git add -N . && git diff HEAD` is graded; an unedited tree always scores 0.\\n- Never create, edit, or delete test files (`test_*.py`, `*_test.py`, `tests/`, `test/`, `testing/`) or runner configs (`pytest.ini`, `conftest.py`, `pyproject.toml`, `setup.cfg`, `tox.ini`, `sitecustomize.py`, `usercustomize.py`).\\n- Put scratch scripts and logs strictly under `/tmp` via `run_command`; any untracked file inside `/workspace` enters your graded patch.\\n- Never run a full test suite (`pytest`, `pytest .`, `unittest discover`). Target one test file with `-x -q -k`, saving output to `/tmp`: `python3 -m pytest tests/path/test_x.py -x -q -k kw > /tmp/t.log 2>&1; tail -n 35 /tmp/t.log` (`run_command` keeps only the first 5,000 chars).\\n- Never inspect outside `/workspace` or install packages (offline container).\\n\\n# CALL PLAN\\n- Calls 1-8: locate and inspect exact target source + caller/test lines.\\n- Calls 9-12: edit SOURCE code. You MUST make at least one source edit by call 12 even if uncertain; an imperfect patch can be refined, pure reading scores 0.\\n- Calls 13-20: run `/tmp` repro + one targeted test file; fix edge cases across all parallel code paths.\\n- Call 21+: stop exploring, verify `git status --short && git diff`, and call `submit_patch`.\\n\\n# WORKFLOW\\n1. **Spec (no tool call)**: List exact files/symbols, error types/messages, return values, and edge cases (`None`, empty, unicode, bounds, defaults, async).\\n2. **Locate**: Start with paths mentioned in the issue. Batch known fixed-string searches in one `run_command` using `git grep -n -F` and `git ls-files` (`head -n 30`). Read 40-120 line slices via `read_file` or `sed -n \'S,Ep\'`, including one caller or sibling test. Keep brief plain-text notes between turns so key facts survive context compaction.\\n3. **Edit**: Use `edit_file` with a short unique `old_string` verbatim from the file. Never repeat an identical failed edit; after 2 match failures, reread exact line numbers and apply one atomic Python replacement via `run_command` quoted heredoc asserting `content.count(old) == 1`. Propagate fixes across ALL parallel implementations (sync/async, `Client`/`AsyncClient`, alternate backends, typing overloads, `__init__.py`/`__all__`, `docs_src/`). Keep unrelated code and signatures intact.\\n4. **Verify**: Run `python3 -m py_compile <edited_file>`, a `/tmp/repro.py` script, and the single closest test file via `/tmp/t.log`. Ignore pre-existing failures from missing optional fixtures or unrelated modules.\\n5. **Submit**: Run `git status --short && git diff`, delete any stray `/workspace` files, revert accidental non-source edits with `git checkout --`, then call `submit_patch` and confirm `patch_size > 0` and `files_changed > 0`. Finish with a one-sentence summary.\\n\\n# DECISION RULES\\n- Ambiguous spec: follow the most literal interpretation and act; never stall.\\n- Stuck on empty searches: try one alternate symbol, then apply your best-reasoned small fix. Never conclude no edit is needed.\\n\\n# Issue\\n{problem_description}\\n"\n}')

for rel_path, content in sorted(SUBMISSION_FILES.items()):
    dest = BUNDLE / rel_path
    dest.parent.mkdir(parents=True, exist_ok=True)
    dest.write_bytes(content.replace("\r\n", "\n").encode("utf-8"))
    print(f"Wrote {rel_path:24s} ({dest.stat().st_size:,} bytes)")


## 2. Structural, Schema & ADK Placeholder Validation


In [ ]:
ROOT_CONFIGS = ("agent.yaml", "agent.yml", "root_agent.yaml", "root_agent.yml")
ALLOWED_SUFFIXES = {".yaml", ".yml", ".md", ".txt", ".py", ".json", ".safetensors"}
ALLOWED_MODELS = (
    "gemma-4-31b-it-qat-w4a16-ct",
    "gemma-4-12b-it-qat-w4a16-ct",
    "gemma-4-e4b-it-qat-w4a16-ct",
    "gemma-4-e2b-it-qat-w4a16-ct",
)
ALLOWED_STATE_VARS = {"problem_description", "hints"}

yaml.SafeLoader.add_constructor("!include", lambda loader, node: loader.construct_scalar(node))

files = sorted(p for p in BUNDLE.rglob("*") if p.is_file())
roots = [name for name in ROOT_CONFIGS if (BUNDLE / name).is_file()]
assert len(roots) == 1, f"Expected 1 root config, found {roots}"
assert not any(p.is_symlink() for p in BUNDLE.rglob("*")), "Symlinks rejected"
assert all(p.suffix in ALLOWED_SUFFIXES for p in files), f"Bad suffix: {[p.name for p in files if p.suffix not in ALLOWED_SUFFIXES]}"
assert sum(p.stat().st_size for p in files) < 3 * 2**30, "Bundle exceeds 3 GiB"

declared_models = set()
for cfg in (p for p in files if p.suffix in (".yaml", ".yml")):
    text = cfg.read_text(encoding="utf-8")
    data = yaml.safe_load(text) or {}
    for inc in re.findall(r"!include\s+(\S+)", text):
        inc_path = (cfg.parent / inc).resolve()
        assert BUNDLE.resolve() in inc_path.parents and inc_path.is_file(), f"{cfg.name}: bad include {inc}"
    assert "adapter" not in data, "Unexpected adapter key"
    if "model" in data:
        declared_models.add(str(data["model"]).strip())

assert len(declared_models) == 1, f"Single model rule violated: {declared_models}"
model = next(iter(declared_models))
assert model in ALLOWED_MODELS, f"Unsupported model: {model}"

for md_file in (p for p in files if p.suffix in (".md", ".txt")):
    placeholders = set(re.findall(r"\{([^}]*)\}", md_file.read_text(encoding="utf-8")))
    unknown = placeholders - ALLOWED_STATE_VARS
    assert not unknown, f"{md_file.name}: unknown ADK state vars {unknown}"

print(f"Schema OK: root={roots[0]} | files={len(files)} | model={model}")


## 3. Official Harness Compilation Check (`adk_submission` + `swegemma`)


In [ ]:
HARNESS_WHEELS = ("google_adk-", "google_genai-", "adk_submission-", "adk_eval_core-", "swegemma-")
PROVIDER_PREFIXES = ("openai/", "google/", "hosted_vllm/", "custom/")

def install_harness() -> None:
    found = sorted(Path("/kaggle/input").glob("**/adk_submission-*.whl")) if Path("/kaggle/input").exists() else []
    if not found:
        raise FileNotFoundError("metric/gemma-4-developer-agent-wheelhouse not attached")
    wh_dir = found[0].parent
    wheels = [str(w) for w in sorted(wh_dir.glob("*.whl")) if w.name.startswith(HARNESS_WHEELS)]
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-deps", *wheels], check=True)

def run_command(command: str) -> str: ...
def read_file(filepath: str, start_line: int | None = None, end_line: int | None = None) -> str: ...
def write_file(filepath: str, content: str) -> str: ...
def edit_file(filepath: str, old_string: str, new_string: str, allow_multiple: bool = False) -> str: ...
def submit_patch() -> str: ...
def get_status() -> str: ...

def _norm_model(name: str) -> str:
    name = name.strip().lower()
    for p in PROVIDER_PREFIXES:
        name = name.removeprefix(p)
    return name

ON_KAGGLE = Path("/kaggle/input").exists()
if not ON_KAGGLE:
    try:
        import adk_submission  # noqa: F401
        HAS_HARNESS = True
    except ImportError as exc:
        HAS_HARNESS = False
        print(f"Harness wheel check skipped offline ({exc!r}); structural validation passed.")
else:
    try:
        import adk_submission  # noqa: F401
    except ImportError:
        install_harness()
    HAS_HARNESS = True

if HAS_HARNESS:
    from adk_submission import ModelRegistry, compile_submission, validate_directory
    from adk_submission.discovery import discover_declared_models
    from google.adk.models.lite_llm import LiteLlm
    from swegemma.config import build_submission_limits

    limits, constraints = build_submission_limits()
    validate_directory(BUNDLE, limits)
    print("validate_directory: PASSED")
    declared = discover_declared_models(BUNDLE, normalize_fn=_norm_model)
    assert declared == {model}, f"Mismatch in declared models: {declared}"
    tools = {
        fn.__name__: fn
        for fn in (run_command, read_file, write_file, edit_file, submit_patch, get_status)
    }
    models = ModelRegistry()
    models.register(model, LiteLlm(model=f"openai/{model}", api_base="http://127.0.0.1:9/v1", api_key="EMPTY"))
    compiled_agent = compile_submission(BUNDLE, tools, models, limits=limits, generation_constraints=constraints)
    print(f"compile_submission: PASSED (root_agent={compiled_agent.name!r}, tools={len(tools)})")


## 5. Emit Clean Files-Only `submission.zip`


In [ ]:
ZIP_PATH = WORKING / "submission.zip"
with zipfile.ZipFile(ZIP_PATH, "w") as zf:
    for p in sorted(f for f in BUNDLE.rglob("*") if f.is_file()):
        info = zipfile.ZipInfo(p.relative_to(BUNDLE).as_posix(), date_time=(1980, 1, 1, 0, 0, 0))
        info.compress_type = zipfile.ZIP_DEFLATED
        info.external_attr = 0o100644 << 16
        zf.writestr(info, p.read_bytes())

with zipfile.ZipFile(ZIP_PATH, "r") as zf:
    assert zf.testzip() is None
    names = zf.namelist()
    assert "agent.yaml" in names and not any(n.endswith("/") for n in names)

shutil.rmtree(BUNDLE)
digest = hashlib.sha256(ZIP_PATH.read_bytes()).hexdigest()
print(f"Submission ready: {ZIP_PATH} ({ZIP_PATH.stat().st_size:,} bytes) | SHA-256: {digest}")
for n in names:
    print(" ", n)
